In [1]:
# === Imports and core helpers (Cell 1) ===
import re
from pathlib import Path
import csv
from collections import Counter
from difflib import SequenceMatcher
import datetime
import os
import threading
import concurrent.futures

# --- gibberish detector ---
def is_gibberish_line(line: str, nonlatin_thresh: float = 0.3, short_token_thresh: float = 0.6) -> bool:
    if not line or line.strip() == "":
        return True
    sample = line.replace('\f', ' ').strip()
    allowed_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9\s\'’\-]")
    nonlatin = sum(1 for ch in sample if not allowed_re.match(ch))
    nonlatin_ratio = nonlatin / max(1, len(sample))
    if nonlatin_ratio > nonlatin_thresh:
        return True
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ']+", sample)
    if not tokens:
        return True
    num_tokens = len(tokens)
    short_tokens = sum(1 for t in tokens if len(t) <= 2)
    if num_tokens >= 3 and (short_tokens / num_tokens) >= short_token_thresh:
        return True
    upper_fragments = sum(1 for t in tokens if t.isupper() and len(t) <= 3)
    if num_tokens >= 2 and (upper_fragments / num_tokens) > 0.6:
        return True
    letters = [ch for ch in sample if re.match(r"[A-Za-zÀ-ÖØ-öø-ÿ]", ch)]
    if len(letters) >= 10:
        vowels = sum(1 for ch in letters if ch.lower() in 'aeiou')
        if (vowels / len(letters)) < 0.18:
            return True
    if re.search(r'(.)\1{4,}', sample):
        return True
    return False

# --- core cleaning ---
def common_cleaning(text: str, remove_gibberish: bool = True, log_dir: Path = None,
                    nonlatin_thresh: float = 0.3, short_token_thresh: float = 0.6) -> str:
    text = text.replace('\f', '\n')
    text = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ0-9\s\.?\!\"',;:\-\(\)]", ' ', text)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+([\.!?;:,])", r"\1", text)

    if remove_gibberish:
        kept_lines = []
        dropped = []
        for ln in text.splitlines():
            if is_gibberish_line(ln, nonlatin_thresh=nonlatin_thresh, short_token_thresh=short_token_thresh):
                dropped.append(ln)
            else:
                kept_lines.append(ln)
        if log_dir and dropped:
            log_dir.mkdir(parents=True, exist_ok=True)
            with open(log_dir / 'removed_gibberish_lines.txt', 'w', encoding='utf-8') as f:
                for d in dropped:
                    f.write(d + '\n')
            print(f'[LOG] Wrote {len(dropped)} removed lines to {log_dir / "removed_gibberish_lines.txt"}')
        text = '\n'.join(kept_lines)

    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text

# --- whitelist builder ---

def build_xh_whitelist_from_dir(source_dir: Path, top_n: int = 500, min_freq: int = 3, include_seed: bool = True):
    if not source_dir.exists():
        print(f"[WHITELIST] source dir {source_dir} not found — falling back to seed list")
        return set(XH_WHITELIST_SEED)

    counter = Counter()
    tok_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")
    files = list(source_dir.glob('*.txt'))
    if not files:
        print(f"[WHITELIST] no .txt files in {source_dir} — falling back to seed list")
        return set(XH_WHITELIST_SEED)

    for p in files:
        try:
            txt = p.read_text(encoding='utf-8')
        except Exception as e:
            print(f"[WHITELIST] could not read {p}: {e}")
            continue
        toks = [t.lower() for t in tok_re.findall(txt)]
        toks = [t for t in toks if len(t) >= 3 and not re.search(r"\d", t)]
        counter.update(toks)

    candidates = [w for w, f in counter.most_common() if f >= min_freq][:top_n]
    whitelist = set(candidates)
    if include_seed:
        whitelist |= set(XH_WHITELIST_SEED)

    if not whitelist:
        print("[WHITELIST] extracted no candidates — falling back to seed list")
        return set(XH_WHITELIST_SEED)

    print(f"[WHITELIST] built whitelist with {len(whitelist)} tokens (source: {source_dir}, files scanned: {len(files)})")
    return whitelist

# --- whitelist helpers ---

def line_has_whitelist_tokens(line: str, whitelist: set, min_hits: int = 1) -> bool:
    toks = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ']+", line.lower())
    hits = sum(1 for t in toks if t in whitelist)
    return hits >= min_hits


def looks_like_xhosa_morphology(line: str) -> bool:
    return bool(re.search(r"\b(?:ndi|si|ni|u|ba|ku|ya|nga)[a-zà-öø-ÿ']{2,}", line.lower()))

print('Core imports and helpers loaded.')


Core imports and helpers loaded.


In [2]:
# === Tweakable parameters (Cell 2) ===
# Toggleable modes and paths — change these before running the notebook
GIBBERISH_REMOVER = True  # set False to disable automatic gibberish-line removal
GIB_NONLATIN_THRESH = 0.25
GIB_SHORT_TOKEN_THRESH = 0.6

# Logging paths
GIB_LOG_DIR = Path.cwd() / "logs"

# === isiXhosa whitelist (seed) ===
XH_WHITELIST_SEED = {
    "kuba","kodwa","na","ke","xa","nxa",
    "mna","wena","yena","thina","nina","abo","lona",
    "ewe","hayi","enkosi","ndiyabulela","ndiyavuya",
    "molo","molweni","sawubona",
    "funda","hamba","bona","thanda","wenza","sebenza","cela","fika","nika",
    "indlu","umntu","abantu","umsebenzi","amanzi","isikolo","imali",
    "ndi","uya","ba","si","ni","ku","sa","ya"
}

# Auto-build configuration for whitelist
AUTO_BUILD_XH_WHITELIST = True
XH_WHITELIST_TOP_N = 500
XH_WHITELIST_MIN_FREQ = 3
XH_WHITELIST_SOURCE_DIR = Path.cwd() / 'final_corpus'

# Batch settings
BATCH_CATEGORY = "general"
RAW_INPUTS_DIR = Path.cwd() / "raw_outputs"
PROCESSED_REGISTRY = Path.cwd() / "logs" / "processed_files.csv"
PROCESS_LOG_DIR = Path.cwd() / "logs" / "process_logs"
ALERT_DIR = Path.cwd() / "logs" / "alerts"
MAX_WORKERS = min(8, (os.cpu_count() or 2))

# Processing mode (set to 'novel', 'drama', or 'drama_sentence')
mode = "novel"

# Build whitelist (the builder function is defined in the imports/functions cell)
if AUTO_BUILD_XH_WHITELIST:
    src = XH_WHITELIST_SOURCE_DIR
    if not src.exists():
        src = Path.cwd() / 'raw_outputs'
    try:
        # build_xh_whitelist_from_dir is defined in the imports/helpers cell (Cell 1)
        XH_WHITELIST = build_xh_whitelist_from_dir(src, top_n=XH_WHITELIST_TOP_N, min_freq=XH_WHITELIST_MIN_FREQ, include_seed=True)
    except NameError:
        # If the builder isn't defined yet (e.g., this cell executed before imports cell), fall back to seed
        XH_WHITELIST = set(XH_WHITELIST_SEED)
else:
    XH_WHITELIST = set(XH_WHITELIST_SEED)

print(f"Parameters set. GIBBERISH_REMOVER={GIBBERISH_REMOVER}, whitelist size (seed or built) will be computed when builder runs.")


[WHITELIST] no .txt files in /home/hlatsiieyhax/DevBlock/godhand_development/inhouse-tools/finalcorpusnator/final_corpus — falling back to seed list
Parameters set. GIBBERISH_REMOVER=True, whitelist size (seed or built) will be computed when builder runs.


In [3]:
# === Mode Selection ====
# Mode is set in the parameters cell; change there if needed

# === Drama Mode Restrictions ===
'''These are words that aren't necessarily speaker labels, etc'''
EXCLUDE_WORDS = {"INDIMA", "IMIBUZO", "KWENGXOXO", "ISINXUMEZELELO", 
                 "ITS", "ID", "PE", "EQ", "OV", "OFUNA", "EMIFUTSHANE",
                 "IMPENDULO", "PHENDULA", "NEEMFUNO"}  # add more if needed

# === File paths ===
TEXT_FILE = "Kungavuka_AbaNguni_ocr.txt"           # Name of the output file
file_dir = Path.cwd() / "raw_outputs" / TEXT_FILE   # Name of the output directory
out_dir = Path.cwd() / "final_corpus"               # Name of the final-corpus directory
out_dir.mkdir(parents=True, exist_ok=True)


In [4]:
# === Preprocess ===
# Use the toggleable gibberish remover mode and thresholds defined earlier
# If `raw_text` isn't already defined (e.g., when running this cell standalone),
# try to read it from the configured `file_dir` path.
if 'raw_text' not in globals():
    try:
        raw_text = file_dir.read_text(encoding='utf-8')
        print(f"[INFO] Read raw_text from {file_dir}")
    except Exception as e:
        raise RuntimeError(f"raw_text not defined and failed to read from {file_dir}: {e}")

cleaned_text = common_cleaning(raw_text,
                               remove_gibberish=GIBBERISH_REMOVER,
                               log_dir=GIB_LOG_DIR if GIBBERISH_REMOVER else None,
                               nonlatin_thresh=GIB_NONLATIN_THRESH,
                               short_token_thresh=GIB_SHORT_TOKEN_THRESH)


[INFO] Read raw_text from /home/hlatsiieyhax/DevBlock/godhand_development/inhouse-tools/finalcorpusnator/raw_outputs/Kungavuka_AbaNguni_ocr.txt
[LOG] Wrote 1704 removed lines to /home/hlatsiieyhax/DevBlock/godhand_development/inhouse-tools/finalcorpusnator/logs/removed_gibberish_lines.txt


In [5]:
def process_drama(text: str):
    """
    Process text in drama mode:
    - Detects and normalizes speaker names.
    - Groups their dialogue until an empty line or next speaker.
    - Splits narrative (non-speaker) blocks into one sentence per line.
    """
    name_map, canonical_names = discover_and_normalize_characters(text)

    output_lines = []
    buffer = []
    current_speaker = None
    current_dialogue = []

    def flush_narrative(buf):
        if not buf:
            return []
        joined = " ".join(buf)
        sentences = re.split(r'(?<=[.!?])\s+', joined)
        return [s.strip() for s in sentences if s.strip()]

    for line in text.splitlines():
        raw_line = line
        line = line.strip()

        # --- empty line: close speaker dialogue ---
        if not line:
            if current_speaker and current_dialogue:
                output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
                current_speaker = None
                current_dialogue = []
            continue

        # --- speaker line ---
        match = re.match(r"^([A-Z]+)\s*:?(.*)", line)
        if match:
            raw_name = re.sub(r"[^\w]", "", match.group(1))
            if raw_name in name_map:
                # flush any buffered narrative before new speaker
                if buffer:
                    output_lines.extend(flush_narrative(buffer))
                    buffer = []

                # flush old speaker
                if current_speaker and current_dialogue:
                    output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
                    current_dialogue = []

                # start new speaker
                current_speaker = name_map[raw_name]
                remainder = match.group(2).strip()
                if remainder:
                    current_dialogue.append(remainder)
                continue

        # --- non-speaker text ---
        if current_speaker:
            current_dialogue.append(line)
        else:
            buffer.append(line)

    # --- flush leftovers ---
    if current_speaker and current_dialogue:
        output_lines.append(f"{current_speaker.upper()} : {' '.join(current_dialogue).strip()}")
    if buffer:
        output_lines.extend(flush_narrative(buffer))

    return "\n".join(output_lines), canonical_names

In [6]:
def process_drama_sentence_mode(text: str) -> str:
    """
    Processes drama-like text by:
    - Removing speaker labels entirely
    - Splitting everything into sentences
    - Returning clean sentence-by-sentence text
    """
    # 1. Remove speaker labels (uppercase names, optional numbers, colon, etc.)
    text_no_labels = re.sub(r'^[A-ZÀ-Ý][A-ZÀ-Ý0-9\s\-]*\s?:\s?', '', text, flags=re.MULTILINE)

    # 2. Flatten text but keep paragraph breaks (double newlines)
    text_flat = re.sub(r'\n{2,}', '\n\n', text_no_labels)  # keep double newlines
    text_flat = re.sub(r'\n', ' ', text_flat)  # replace single newlines with spaces

    # 3. Split into sentences (keep punctuation)
    sentences = re.split(r'(?<=[.!?])\s+', text_flat)

    # 4. Clean up each sentence
    sentences = [s.strip() for s in sentences if s.strip()]

    # 5. Return each sentence on its own line
    return '\n'.join(sentences)

In [7]:
def process_novel(text: str):
    """Process text in novel mode: one sentence per line."""
    text = re.sub(r"\n+", " ", text)
    sentences = re.split(r'([\.!?]["\']?)', text)

    cleaned_sentences = []
    current = ""
    for part in sentences:
        current += part.strip() + " "
        if re.fullmatch(r'[\.!?]["\']?', part.strip()):
            cleaned_sentences.append(current.strip())
            current = ""
    if current.strip():
        cleaned_sentences.append(current.strip())

    return "\n".join(cleaned_sentences)

In [8]:
# === Batch Processing (parallel) ===
# This cell walks `RAW_INPUTS_DIR` (default: raw_outputs), processes each text file
# using the cleaning + mode pipeline, logs each run, and keeps a registry so files
# aren't processed twice. Uses ThreadPoolExecutor for parallelism.

# Ensure directories exist
PROCESS_LOG_DIR.mkdir(parents=True, exist_ok=True)
ALERT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_REGISTRY.parent.mkdir(parents=True, exist_ok=True)

# registry lock for thread-safety
_registry_lock = threading.Lock()

# Helper to check registry
def is_already_processed(fp: Path) -> bool:
    if not PROCESSED_REGISTRY.exists():
        return False
    with _registry_lock:
        with open(PROCESSED_REGISTRY, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                parts = line.split(',')
                if parts[0] == str(fp):
                    return True
    return False

# Append an entry to registry
def register_processed(fp: Path, out_path: Path, preserved_ratio: float, dropped_lines: int):
    header = "filepath,processed_at,out_path,preserved_ratio,dropped_lines\n"
    now = datetime.datetime.utcnow().isoformat()
    with _registry_lock:
        write_header = not PROCESSED_REGISTRY.exists()
        with open(PROCESSED_REGISTRY, 'a', encoding='utf-8') as f:
            if write_header:
                f.write(header)
            f.write(f"{fp},{now},{out_path},{preserved_ratio:.4f},{dropped_lines}\n")

# Worker for a single file
def process_file(fp: Path):
    stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    logfile = PROCESS_LOG_DIR / f"{BATCH_CATEGORY}_{stamp}_{fp.stem}.log"

    if is_already_processed(fp):
        with open(logfile, 'w', encoding='utf-8') as lg:
            lg.write(f"SKIP: {fp} already processed\n")
        return

    raw_text = fp.read_text(encoding='utf-8')
    total_chars = len(raw_text)
    total_lines = len(raw_text.splitlines())

    # Run base cleaning (this may drop gibberish lines), but preserve lines matching whitelist/morphology
    # We run common_cleaning with remove_gibberish=False to get initial normalization,
    # then apply per-line removal while respecting whitelist/morph checks.
    normalized = common_cleaning(raw_text, remove_gibberish=False)
    lines = normalized.splitlines()

    kept_lines = []
    dropped_lines_list = []
    for ln in lines:
        if not GIBBERISH_REMOVER:
            kept_lines.append(ln)
            continue
        # Keep if whitelist token found or morphological pattern detected
        if line_has_whitelist_tokens(ln, XH_WHITELIST, min_hits=1) or looks_like_xhosa_morphology(ln):
            kept_lines.append(ln)
            continue
        # otherwise use gibberish detector
        if is_gibberish_line(ln, nonlatin_thresh=GIB_NONLATIN_THRESH, short_token_thresh=GIB_SHORT_TOKEN_THRESH):
            dropped_lines_list.append(ln)
        else:
            kept_lines.append(ln)

    cleaned_text = '\n'.join(kept_lines)
    after_clean_line_count = len(kept_lines)
    dropped_count = len(dropped_lines_list)

    # Further processing based on mode
    if mode == "drama":
        processed_text, characters = process_drama(cleaned_text)
    elif mode == "drama_sentence":
        processed_text = process_drama_sentence_mode(cleaned_text)
        characters = None
    else:
        processed_text = process_novel(cleaned_text)
        characters = None

    out_stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    out_name = f"{BATCH_CATEGORY}_{out_stamp}_{fp.stem}_cleaned.txt"
    out_path = out_dir / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(processed_text, encoding='utf-8')

    # Preserve ratio: use chars in processed_text vs raw_text (fallback to cleaned)
    preserved_chars = len(processed_text)
    preserved_ratio = preserved_chars / max(1, total_chars)

    # Per-file logging
    with open(logfile, 'w', encoding='utf-8') as lg:
        lg.write(f"FILE: {fp}\n")
        lg.write(f"OUT : {out_path}\n")
        lg.write(f"MODE: {mode}\n")
        lg.write(f"TOTAL_CHARS: {total_chars}\n")
        lg.write(f"PRESERVED_CHARS: {preserved_chars}\n")
        lg.write(f"PRESERVED_RATIO: {preserved_ratio:.4f}\n")
        lg.write(f"TOTAL_LINES: {total_lines}\n")
        lg.write(f"AFTER_CLEAN_LINES: {after_clean_line_count}\n")
        lg.write(f"DROPPED_LINES: {dropped_count}\n")
        if characters is not None:
            lg.write(f"CHARACTERS_FOUND: {list(characters)[:20]}\n")
        lg.write("\n--- SAMPLE (first 30 lines of output) ---\n")
        for i,ln in enumerate(processed_text.splitlines()[:30]):
            lg.write(f"{i+1:03d}: {ln}\n")

    # If preserved <= 25%, write an alert file for review
    if preserved_ratio <= 0.25:
        alertf = ALERT_DIR / f"{BATCH_CATEGORY}_{out_stamp}_{fp.stem}_low_preserve.log"
        with open(alertf, 'w', encoding='utf-8') as af:
            af.write(f"LOW PRESERVE ALERT\nFILE: {fp}\nOUT: {out_path}\nPRESERVED_RATIO: {preserved_ratio:.4f}\n")
            af.write(f"TOTAL_CHARS: {total_chars}\nPRESERVED_CHARS: {preserved_chars}\nDROPPED_LINES: {dropped_count}\n\n")
            af.write("--- First 500 characters of cleaned output ---\n")
            af.write(processed_text[:500])

    # Save removed gibberish lines (append) for debugging
    if dropped_count:
        GIB_LOG_DIR.mkdir(parents=True, exist_ok=True)
        with open(GIB_LOG_DIR / 'removed_gibberish_lines.txt', 'a', encoding='utf-8') as gf:
            gf.write(f"# {fp} --- {out_stamp}\n")
            for d in dropped_lines_list:
                gf.write(d + '\n')

    # Register processed
    register_processed(fp, out_path, preserved_ratio, dropped_count)

    print(f"Processed {fp.name} -> {out_path.name} (preserved {preserved_ratio:.1%})")

# Batch run in parallel
candidates = sorted(RAW_INPUTS_DIR.glob('*.txt'))
if not candidates:
    print(f"No .txt files found in {RAW_INPUTS_DIR}")
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_file, fpath) for fpath in candidates]
        for fut in concurrent.futures.as_completed(futures):
            try:
                fut.result()
            except Exception as e:
                err_log = PROCESS_LOG_DIR / f"{BATCH_CATEGORY}_error_{datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.log"
                with open(err_log, 'w', encoding='utf-8') as ef:
                    ef.write(f"ERROR processing: {e}\n")
                print(f"ERROR: see {err_log}")

print('Batch run complete.')


Processed Kungavuka_AbaNguni_ocr.txt -> general_20251013_021633_Kungavuka_AbaNguni_ocr_cleaned.txt (preserved 100.5%)
Batch run complete.
